# Module 04: Training Pipelines & Hyperparameter Tuning

**What you'll learn:**
- Why training should be a pipeline, not a notebook
- Time-series cross-validation (and why regular CV is dangerous)
- Automated hyperparameter tuning with Optuna
- Running end-to-end training pipelines

**Time:** ~1.5 hours

## 1. From Notebook to Pipeline

So far, you've trained models manually:
1. Load data... (copy-paste code)
2. Engineer features... (copy-paste more code)
3. Train model... (tweak parameters by hand)
4. Evaluate... (hope you didn't make a mistake)

This is fine for **experimentation**. But for **production**, you need:

- **Reproducibility**: Anyone can re-run and get the same result
- **Automation**: Schedule retraining without human intervention
- **Reliability**: Each step validates before moving to the next
- **Auditability**: You know exactly what happened in each run

A **training pipeline** chains all these steps into a single, automated workflow.

## 2. Time-Series Cross-Validation

### Why Regular K-Fold CV is WRONG for Time Series

Regular k-fold CV randomly shuffles data into folds. This means **future data leaks into training**:

```
Regular K-Fold (WRONG for time series):
  Fold 1: Train on [Jan, Mar, Jun, Oct]  Test on [Apr]  ← Mar comes AFTER Apr!
  Fold 2: Train on [Jan, Apr, Jun, Oct]  Test on [Mar]  ← Jun comes AFTER Mar!
```

### Correct: Expanding Window CV

```
Expanding Window (CORRECT):
  Fold 1: Train [Jan-Jun]        Test [Jul-Aug]
  Fold 2: Train [Jan-Aug]        Test [Sep-Oct]   ← train GROWS
  Fold 3: Train [Jan-Oct]        Test [Nov-Dec]
```

The training set always comes BEFORE the test set. No future data leaks in.

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import matplotlib.pyplot as plt
from energy_forecast.training.cross_validation import TimeSeriesCV

# Create sample data
X = np.random.randn(1000, 5)
y = np.random.randn(1000)

cv = TimeSeriesCV(n_splits=5, strategy='expanding_window', gap=24)

print('Expanding Window Cross-Validation Splits:')
print(f"{'Fold':<6} {'Train Range':<25} {'Test Range':<25} {'Gap OK?'}")
print('-' * 70)
for fold, (train_idx, test_idx) in enumerate(cv.split(X)):
    gap = test_idx[0] - train_idx[-1]
    print(f"{fold:<6} [{train_idx[0]:>4} : {train_idx[-1]:>4}] ({len(train_idx):>4} pts)   [{test_idx[0]:>4} : {test_idx[-1]:>4}] ({len(test_idx):>4} pts)   gap={gap}")
    assert train_idx[-1] < test_idx[0], 'DATA LEAKAGE!'

print('\nAll folds verified: no data leakage!')

## 3. The Trainer Class

The `Trainer` class connects everything:
- Takes a model + evaluator + config
- Starts an MLflow run
- Trains the model
- Evaluates on train and validation sets
- Logs everything automatically

Let's see it in action:

In [ ]:
from energy_forecast.data.synthetic import SyntheticDataGenerator
from energy_forecast.features.engineering import FeatureEngineer
from energy_forecast.models.xgboost_model import XGBoostForecaster
from energy_forecast.evaluation.evaluator import ModelEvaluator
from energy_forecast.evaluation.metrics import MetricsCalculator
from energy_forecast.training.trainer import Trainer
from sklearn.preprocessing import StandardScaler
import mlflow

mlflow.set_tracking_uri('file:///tmp/mlflow_workshop')

# Prepare data
gen = SyntheticDataGenerator(num_buildings=3, start_date='2023-01-01', end_date='2023-12-31', random_seed=42)
df = gen.generate()
engineer = FeatureEngineer()
df = engineer.create_time_features(df)
df = engineer.create_lag_features(df, 'energy_demand_kwh', [1, 2, 3, 24])
df = engineer.create_rolling_features(df, 'energy_demand_kwh', [6, 24])
df = engineer.create_weather_features(df)
df = engineer.create_calendar_features(df)
df = df.dropna()

feature_names = engineer.get_feature_names(df)
X = df[feature_names].values; y = df['energy_demand_kwh'].values
n = len(X)
X_train, y_train = X[:int(n*0.7)], y[:int(n*0.7)]
X_val, y_val = X[int(n*0.7):int(n*0.85)], y[int(n*0.7):int(n*0.85)]
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

print(f'Data ready: {X_train_s.shape[0]} train, {X_val_s.shape[0]} val samples')

In [ ]:
# Use the Trainer
model = XGBoostForecaster(n_estimators=300, max_depth=6, learning_rate=0.05)
evaluator = ModelEvaluator()
trainer = Trainer(model=model, evaluator=evaluator, config={'experiment_name': 'pipeline-workshop'})

result = trainer.train(X_train_s, y_train, X_val_s, y_val)

print(f'\nRun ID: {result.run_id}')
print(f'\nTraining metrics:')
for k, v in result.metrics.items():
    print(f'  {k}: {v:.4f}')
if result.val_metrics:
    print(f'\nValidation metrics:')
    for k, v in result.val_metrics.items():
        print(f'  {k}: {v:.4f}')
print('\nAll params, metrics, and model artifact logged to MLflow automatically!')

## 4. Hyperparameter Tuning with Optuna

### What is Hyperparameter Tuning?

Models have settings you choose BEFORE training (`learning_rate`, `max_depth`, etc.). Finding the best settings manually is:
- Tedious (hundreds of combinations)
- Biased (you try what you think will work)
- Slow (you forget what you already tried)

**Optuna** automates this. It's like a smart student who:
1. Tries a set of hyperparameters
2. Sees how well it did
3. Learns which parameter regions are promising
4. Focuses the next trial on those regions

This is called **Bayesian optimization** — much smarter than random search.

In [ ]:
from energy_forecast.training.hyperparameter import HyperparameterSearcher

searcher = HyperparameterSearcher(
    model_class=XGBoostForecaster,
    X_train=X_train_s, y_train=y_train,
    X_val=X_val_s, y_val=y_val,
    config={'metric': 'rmse'}
)

result = searcher.search(n_trials=15, timeout=120)

print(f'Best RMSE: {result.best_metric:.4f}')
print(f'\nBest hyperparameters:')
for k, v in result.best_params.items():
    print(f'  {k}: {v}')

# Plot optimization history
fig, ax = plt.subplots(figsize=(10, 5))
values = [t['value'] for t in result.all_trials if t['value'] is not None]
ax.plot(values, 'o-', color='steelblue')
ax.set_xlabel('Trial')
ax.set_ylabel('RMSE')
ax.set_title('Hyperparameter Optimization History')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f'\nOptuna explored {len(result.all_trials)} configurations automatically!')

## 5. The End-to-End Training Pipeline

Now let's run everything in ONE call — data generation through model evaluation:

In [ ]:
from energy_forecast.pipelines.training_pipeline import TrainingPipeline

pipeline = TrainingPipeline(
    data_config={
        'synthetic': {'num_buildings': 3, 'start_date': '2023-01-01', 'end_date': '2023-12-31', 'random_seed': 42},
        'processing': {'lag_features': [1,2,3,24], 'rolling_windows': [6,24], 'train_ratio': 0.7, 'val_ratio': 0.15, 'test_ratio': 0.15}
    },
    model_type='xgboost'
)

result = pipeline.run()

print('Pipeline Complete!')
print(f'Data shape: {result.data_shape}')
print(f'Features used: {len(result.feature_names)}')
print(f'\nTest Metrics:')
for k, v in result.metrics.items():
    print(f'  {k}: {v:.4f}')

## 6. Running as a Script

In production, you don't open a notebook. You run a command:

```bash
python -m energy_forecast.pipelines.training_pipeline
```

Or with Make:
```bash
make train
```

This is what Airflow DAGs and CI/CD pipelines will call — we'll see that in Notebooks 07 and 08.

## 7. Exercises

1. Run the pipeline with `model_type='linear'` and compare results to xgboost
2. Use `HyperparameterSearcher` with `LinearForecaster` — does alpha matter much?
3. Try `TimeSeriesCV` with `strategy='sliding_window'` — how do results differ?

## 8. Key Takeaways

- **Pipelines** make training reproducible and automated
- **Time-series CV** prevents data leakage (never use random k-fold!)
- **Optuna** finds better hyperparameters than manual tuning
- The **end-to-end pipeline** is what gets deployed in production
- Command-line execution enables automation (Airflow, CI/CD)

**Next: [Notebook 05 - Model Serving](./05_model_serving.ipynb)**